In [ ]:
# Import necessary libraries
import os
import re
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from sqlalchemy import create_engine, text
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time
from pathlib import Path
from urllib.parse import urljoin
from selenium.webdriver.chrome.options import Options
import pandas as pd
load_dotenv()

In [ ]:
url = "https://books.toscrape.com/catalogue/page-1.html"

r = requests.get(url)
print(r.status_code)

### WebDriver Initialization

In [ ]:
def make_driver():
    chrome_options = Options()

    # "--headless=new" works better with newer Chrome versions
    #chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    # Selenium Manager handles ChromeDriver automatically
    return webdriver.Chrome(options=chrome_options)

### Books Extraction

In [ ]:
# Lists to store the extracted data
book_names = []
prices = []
in_stocks = []
ratings = []
book_urls = []
book_images = []
categories = []
source_websites = []
scraped_at = []


SOURCE_WEBSITE = "https://books.toscrape.com/"
BOOK_BASE_URL = urljoin(SOURCE_WEBSITE, "catalogue/")
current_url = urljoin(BOOK_BASE_URL, "page-1.html")
page_number = 1


detail_driver = make_driver()
try:
    while current_url:
        url = current_url
        detail_driver.get(url)
        WebDriverWait(detail_driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "article.product_pod"))
        )

        # Parse the catalogue page currently displayed in Chrome.
        soup = BeautifulSoup(detail_driver.page_source, "html.parser")

        # Locate each book card in the catalogue.
        books = soup.select("article.product_pod")

        # Save the next page URL before visiting book detail pages.
        next_element = soup.select_one("li.next a")
        next_page_url = (
            urljoin(url, next_element.get("href"))
            if next_element and next_element.get("href")
            else None
        )

        for book in books:
            # Extract book name
            title_element = book.select_one("h3 a")
            price_element = book.select_one("p.price_color")
            rating_element = book.select_one("p.star-rating")

            book_name = (
                title_element.get("title") if title_element else None
            )

            # Extract prices
            price = (
                price_element.get_text(strip=True)
                .replace("£", "")
                .replace(",", "")
                if price_element
                else None
            )

            # The stock quantity is extracted from the detail page below.
            in_stock = None

            # Extract rating
            rating_classes = (
                rating_element.get("class", []) if rating_element else []
            )
            rating = next(
                (value for value in rating_classes if value != "star-rating"),
                None,
            )

            # Extract book URL
            book_url = (
                urljoin(BOOK_BASE_URL, title_element["href"])
                if title_element and title_element.get("href")
                else None
            )

            # Category and full availability are on the book detail page.
            category = None
            if book_url:
                detail_driver.get(book_url)
                WebDriverWait(detail_driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "ul.breadcrumb"))
                )
                detail_soup = BeautifulSoup(detail_driver.page_source, "html.parser")
                category_element = detail_soup.select_one(
                    "ul.breadcrumb li:nth-of-type(3) a"
                )
                category = (
                    category_element.get_text(strip=True)
                    if category_element
                    else None
                )

                # Extract the quantity, for example: In stock (22 available).
                for row in detail_soup.select("table.table-striped tr"):
                    heading = row.select_one("th")
                    value = row.select_one("td")
                    if heading and heading.get_text(strip=True) == "Availability":
                        availability_text = (
                            value.get_text(" ", strip=True) if value else ""
                        )
                        match = re.search(r"\d+", availability_text)
                        in_stock = int(match.group()) if match else None
                        break

            # Extract book image URL
            book_image = (
                urljoin(BOOK_BASE_URL, book.select_one("img")["src"])
                if book.select_one("img")
                else None
            )

            # Append the extracted data to the respective lists
            book_names.append(book_name)
            prices.append(price)
            in_stocks.append(in_stock)
            ratings.append(rating)
            categories.append(category)
            book_urls.append(book_url)
            book_images.append(book_image)
            source_websites.append(SOURCE_WEBSITE)
            scraped_at.append(pd.Timestamp.now(tz="UTC"))

        # print(f"Scraped page {page_number}: {len(book_names)} books collected")
        current_url = next_page_url
        page_number += 1
finally:
    detail_driver.quit()

# Create a DataFrame from the extracted data
data = {
    "book_names": book_names,
    "availability": in_stocks,
    "ratings": ratings,
    "prices": prices,
    "categories": categories,
    "book_urls": book_urls,
    "book_images": book_images,
    "source_website": source_websites,
    "scraped_at": scraped_at,
}

books_df = pd.DataFrame(data)

### Transformation

In [ ]:
# Save the DataFrame to a CSV file
books_df.to_csv("../data/raw_data/books_data.csv", index=False)

In [ ]:
# Convert price and availability values to numbers.
books_df["prices"] = pd.to_numeric(books_df["prices"], errors="coerce")
books_df["availability"] = pd.to_numeric(
    books_df["availability"], errors="coerce"
).astype("Int64")

In [ ]:
# Convert the rating column to numeric values for easier analysis
rating_mapping = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5,
}
books_df["ratings"] = books_df["ratings"].map(rating_mapping).astype("Int64")

In [ ]:
# Store all categories discovered in the book breadcrumbs.
books_df["categories"] = books_df["categories"].astype("category")

# Keep the source as text and normalize scrape timestamps to UTC.
books_df["source_website"] = (
    books_df["source_website"].astype("string").str.strip()
)
books_df["scraped_at"] = pd.to_datetime(
    books_df["scraped_at"], errors="coerce", format="mixed", utc=True
)

# Validate the cleaned data before saving or loading it.
required_columns = [
    "book_names",
    "availability",
    "ratings",
    "prices",
    "categories",
    "book_urls",
    "source_website",
    "scraped_at",
]
validation_errors = []
missing_columns = set(required_columns).difference(books_df.columns)

if missing_columns:
    validation_errors.append(
        "missing columns: " + ", ".join(sorted(missing_columns))
    )
elif books_df.empty:
    validation_errors.append("the dataset is empty")
else:
    if books_df[required_columns].isna().any().any():
        validation_errors.append("required fields contain null values")
    if books_df["book_urls"].duplicated().any():
        validation_errors.append("duplicate book URLs were found")
    if not books_df["ratings"].between(1, 5).all():
        validation_errors.append("ratings must be between 1 and 5")
    if (books_df["prices"] < 0).any():
        validation_errors.append("prices cannot be negative")
    if (books_df["availability"] < 0).any():
        validation_errors.append("availability cannot be negative")
    for column in ["book_names", "categories", "book_urls", "source_website"]:
        if books_df[column].astype("string").str.strip().eq("").any():
            validation_errors.append(f"{column} contains blank values")
    if books_df["scraped_at"].dt.tz is None:
        validation_errors.append("scraped_at must use a UTC timezone")

if validation_errors:
    raise ValueError(
        "Data validation failed: " + "; ".join(validation_errors)
    )

print(f"Data validation passed for {len(books_df)} books")

In [ ]:
# Save the cleaned DataFrame to a CSV file
books_df.to_csv("../data/cleaned_data/books_data.csv", index=False)

### Create Database and Load Books

In [ ]:
# Create a database in PostgreSQL and store the data in a table
# Define the database connection parameters
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

In [ ]:
# Create the PostgreSQL database if it does not already exist
admin_engine = create_engine(
    f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/postgres",
    isolation_level="AUTOCOMMIT",
)
with admin_engine.connect() as connection:
    database_exists = connection.execute(
        text("SELECT 1 FROM pg_database WHERE datname = :db_name"),
        {"db_name": db_name},
    ).scalar()
    if not database_exists:
        quoted_db_name = admin_engine.dialect.identifier_preparer.quote_identifier(
            db_name
        )
        connection.exec_driver_sql(f"CREATE DATABASE {quoted_db_name}")

admin_engine.dispose()
engine = create_engine(
    f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)

In [ ]:
# # Create schema and table in the database
with engine.connect() as connection:
    # Create schema if it doesn't exist
    connection.execute(text("CREATE SCHEMA IF NOT EXISTS pagepulse;"))

    # Create table if it doesn't exist
    connection.execute(text("""
        CREATE TABLE IF NOT EXISTS pagepulse.books (
            book_names TEXT ,
            availability INTEGER,
            ratings TEXT,
            prices FLOAT,
            categories TEXT,
            book_urls TEXT,
            book_images TEXT,
            source_website TEXT,
            scraped_at TIMESTAMPTZ
        );
    """))

In [ ]:
# Load the saved cleaned data into the PostgreSQL database
with engine.begin() as connection:
    connection.execute(text("CREATE SCHEMA IF NOT EXISTS pagepulse"))
    books_df.to_sql(
        "books",
        connection,
        schema="pagepulse",
        if_exists="replace",
        index=False,
    )
print("Data loaded into PostgreSQL database successfully.")

### SQL Business Analysis

In [ ]:
sql_task_1 = """
-- 1. Total number of books
SELECT COUNT(*) AS total_books
FROM pagepulse.books;
"""
result = pd.read_sql(sql_task_1, engine)
result

In [ ]:
sql_task_2 = """
-- 2. Average book price rounded to two decimal places
SELECT ROUND(CAST(AVG(prices) AS numeric), 2) AS average_price
FROM pagepulse.books;
"""
result = pd.read_sql(sql_task_2, engine)
result

In [ ]:
sql_task_3 = """
SELECT book_names, prices FROM pagepulse.books
ORDER BY prices DESC
LIMIT 10;
"""
result = pd.read_sql(sql_task_3, engine)
result

In [ ]:
sql_task_4 = """
-- 4. Largest category by number of books
SELECT categories, COUNT(*) AS book_count
FROM pagepulse.books
GROUP BY categories
ORDER BY book_count DESC
LIMIT 1;
"""
result = pd.read_sql(sql_task_4, engine)
result

In [ ]:
sql_task_5 = """
-- 5. Category with the highest average price
SELECT categories, AVG(prices) AS average_price  
FROM pagepulse.books
GROUP BY categories
ORDER BY average_price DESC
LIMIT 1;
"""
result = pd.read_sql(sql_task_5, engine)
result

In [ ]:
sql_task_6 = """
-- 6. Number of books by rating
SELECT ratings, COUNT(*) AS book_count
FROM pagepulse.books
GROUP BY ratings
ORDER BY book_count DESC;
"""
result = pd.read_sql(sql_task_6, engine)
result

In [ ]:
sql_task_7 = """
-- 7. All books currently in stock
SELECT COUNT(*) AS in_stock_books
FROM pagepulse.books;
"""
result = pd.read_sql(sql_task_7, engine)
result

In [ ]:
sql_task_8 = """
-- 8. Books priced above £ 40
SELECT book_names, prices
FROM pagepulse.books
WHERE prices > 40
ORDER BY prices DESC; 
"""
result = pd.read_sql(sql_task_8, engine)
result

In [ ]:
sql_task_9 = """
-- 9. Average rating by category
SELECT categories, ROUND(AVG(ratings), 2) AS average_rating
FROM pagepulse.books
GROUP BY categories
ORDER BY average_rating DESC; 
"""
result = pd.read_sql(sql_task_9, engine)
result

In [ ]:
sql_task_10 = """
-- 10. Categories with more than 20 books
SELECT categories, COUNT(*) AS book_count
FROM pagepulse.books
GROUP BY categories
HAVING COUNT(*) > 20
ORDER BY book_count DESC;
"""
result = pd.read_sql(sql_task_10, engine)
result

### Business Insights

##### 1. The catalogue has substantial category coverage

**Evidence:** The catalogue contains **1,000 books across 50 categories**.

**What it means:** BookSphere Analytics has enough books to compare categories by size, price, rating, and stock, but results for smaller categories may be less reliable.

#### 2. The typical book is priced at about £35

**Evidence:** The average book price is **£35.07**. A total of **403 books (40.3%)** cost more than £40.

**What it means:** The catalogue has a strong mid-to-premium price position. BookSphere Analytics could use £35.07 as an initial pricing benchmark and investigate whether the catalogue needs more affordable products for price-sensitive customers.

#### 3. The highest-priced products are tightly grouped

**Evidence:** *The Perfect Play (Play by Play #1)* is the most expensive book at **£59.99**. The next four most expensive books cost between **£59.90 and £59.98**.

**What it means:** The premium end of the catalogue appears to have a price ceiling near £60. BookSphere Analytics could monitor whether these products also achieve strong ratings or sales before recommending premium promotions.


#### 4. Nonfiction is the largest clearly defined category

**Evidence:** Category Default (152 books), **Nonfiction contains 110 books**, followed by Sequential Art (75). The Add a comment category contains 67 books

**What it means:** Nonfiction is commercially important because it represents a large portion of the usable assortment. The 219 books (Default plus Add a comment) assigned to ambiguous labels should be reviewed because weak classification can reduce the accuracy of category reporting and recommendations.


#### 5. Lower-rated books slightly outweigh highly rated books

**Evidence:** There are **422 one- or two-star books (42.2%)**, compared with **375 four- or five-star books (37.5%)**. One-star books form the largest individual rating group, with 226 titles.

**What it means:** BookSphere Analytics may need to identify categories containing a high share of poorly rated products. These titles could be reviewed for replacement or reduced promotion, although rating results should be combined with sales and customer-engagement data before making assortment decisions.